# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset title and description
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

To ensure unambiguous references, **all entities are referenced by their `@id` fields.**

In [ ]:
print("Available record sets in this dataset:")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"- Name: {rs.name}, @id: {rs.id}")

# For each record set, show fields/columns
for rs in record_sets:
    print(f"\nFields/columns for record set '{rs.name}' (@id: {rs.id}):")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, dataType: {field.data_type})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

The main survey data is often in a record set with a name like `SurveyResults` or similar documentation. We'll extract all record sets found. All references are always by `@id`.

In [ ]:
# Extract all record set @ids
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    df = pd.DataFrame(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for record set {record_set_id} (shape: {df.shape})")

# Print available columns in the main record set
if record_set_ids:
    main_rs_id = record_set_ids[0]  # Choose the first record set for exploration; change as needed
    print(f"\nColumns in record set {main_rs_id}:")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

All fields are referenced by their exact `@id` as defined in the schema.

In [ ]:
import numpy as np

# Select numeric field for analysis
# Use the exact @id for the GAD-7 total score field (replace if correct @id is known from section 2)
# For this notebook, let's say the field @id is 'gad7_total_score' (update as needed)

# List columns for selection (helpful for mapping @id to column)
print("All columns in DataFrame for selection:")
print(dataframes[main_rs_id].columns.tolist())

# Replace below with the exact field @id (column) as found in section 2/output above
numeric_field_id = 'gad7_total_score'  # <--- change if needed
if numeric_field_id in dataframes[main_rs_id].columns:
    threshold = 10
    filtered_df = dataframes[main_rs_id][dataframes[main_rs_id][numeric_field_id] > threshold].copy()
    print(f"Filtered records with '{numeric_field_id}' > {threshold}:")
    display(filtered_df.head())

    # Normalize the GAD-7 score (z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by another field (for example, 'gender' with @id 'gender')
    group_field_id = 'gender'  # Update as per available columns and @id
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nMean '{numeric_field_id}' grouped by '{group_field_id}':")
        display(grouped_df)
else:
    print(f"Field '{numeric_field_id}' not found in columns. Please select available field @id for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution of GAD-7 scores (replace with available @id if needed)
if numeric_field_id in dataframes[main_rs_id].columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(dataframes[main_rs_id][numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f'Distribution of {numeric_field_id} (GAD-7 Total Score)')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Boxplot by group (e.g., gender)
    if group_field_id in dataframes[main_rs_id].columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=dataframes[main_rs_id])
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset offers rich mental health survey data, with demographic and screening variables relevant for research.
- Fields are referenced by `@id` as per the Croissant schema—ensure to refer to the schema for precise field/column selection.
- Example analysis and visualizations explored GAD-7 scores and demographic breakdowns.

**Modify and extend analyses using field and record set `@id`s from the Croissant schema to ensure reproducibility and clarity.**